# 🗺️ TURYA'S ROUTING MODULE TESTING NOTEBOOK

**Test the complete routing system: A* Router + Dispatch Classifier + Hospital Ranker**

---

## 📦 SETUP: Import Modules

In [ ]:
import sys
sys.path.append('../modules')

from routing import NaviRakshaRouter
from routing.a_star_router import AStarRouter
from routing.dispatch_classifier import DispatchClassifier
from routing.hospital_ranker import HospitalRanker

import pandas as pd
import pickle

print("✅ Modules imported successfully!")

## 🛣️ TEST 1: A* Router

In [ ]:
# Load graph
with open('../data/raw/navi_mumbai_road_graph.pkl', 'rb') as f:
    G = pickle.load(f)

router = AStarRouter(G)

# Test route from Vashi to Belapur
source_lat, source_lon = 19.0822, 73.0308  # Vashi
dest_lat, dest_lon = 19.02, 73.04  # Belapur

# Find nearest nodes
source_node = None
dest_node = None
min_dist_s = float('inf')
min_dist_d = float('inf')

for node, data in G.nodes(data=True):
    dist_s = ((data['y'] - source_lat)**2 + (data['x'] - source_lon)**2)**0.5
    if dist_s < min_dist_s:
        min_dist_s = dist_s
        source_node = node
    
    dist_d = ((data['y'] - dest_lat)**2 + (data['x'] - dest_lon)**2)**0.5
    if dist_d < min_dist_d:
        min_dist_d = dist_d
        dest_node = node

print(f"Source node: {source_node}")
print(f"Dest node: {dest_node}")

# Find route
route, eta = router.find_route(source_node, dest_node, hour=14, is_monsoon=False)

if route:
    print(f"✅ Route found! {len(route)} nodes, ETA: {eta:.1f} min")
else:
    print("❌ No route found")

## 🚑 TEST 2: Dispatch Classifier

In [ ]:
dispatcher = DispatchClassifier()

# Test cases
test_cases = [
    ('Critical', 5.2, 'Cardiac'),
    ('High', 3.0, 'Trauma'),
    ('Medium', 1.5, 'Burn'),
    ('Low', 10.0, 'Respiratory')
]

for severity, distance, incident in test_cases:
    ambulance = dispatcher.classify(severity, distance, incident)
    print(f"{severity} {incident} ({distance}km) → {ambulance}")

## 🏥 TEST 3: Hospital Ranker

In [ ]:
hospitals_df = pd.read_csv('../data/raw/hospitals_navi_mumbai.csv')
ranker = HospitalRanker(router, hospitals_df)

ranked = ranker.rank_hospitals(
    patient_lat=19.0822,
    patient_lon=73.0308,
    ambulance_type='ALS',
    hour=14,
    is_monsoon=False,
    max_results=3
)

print("Top Hospitals:")
for h in ranked:
    print(f"  {h['rank']}. {h['hospital_name']} - ETA: {h['eta_min']:.1f} min, Beds: {h['available_beds']}/{h['total_beds']}")

## 🔗 TEST 4: Full Emergency Handling

In [ ]:
routing = NaviRakshaRouter()

emergency_data = {
    'patient_lat': 19.0822,
    'patient_lon': 73.0308,
    'incident_type': 'Cardiac',
    'severity': 'Critical',
    'hour': 14,
    'is_monsoon': False
}

response = routing.handle_emergency(emergency_data)

print("🚨 EMERGENCY RESPONSE:")
print(f"  Ambulance Type: {response['ambulance_type']}")
print(f"  ETA to Hospital: {response['eta_min']:.1f} min")
print(f"  Best Hospital: {response['best_hospital']['name']}")
print(f"  Available Beds: {response['best_hospital']['available_beds']}")

print("\n📋 Alternatives:")
for alt in response['alternatives']:
    print(f"  {alt['name']} - ETA: {alt['eta_min']:.1f} min, Beds: {alt['beds']}")